### ChatBot and RAG Evaluation

### Overview

## A typical RAG evaluation workflow consists of three main steps

1. Creating a dataset with questions and their expected answers
2. Running your RAG in those questions
3. Using evaluators to measures how well your application performed, looking at factors like:

Answer relevance, Answer accuracy, Retrieval quality

### Chatbot evaluation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [9]:
### create DATA Sets
from langsmith import Client
client=Client()

## Define the datasets- these are your test data
dataset_name="Chatbot Evaluation"
dataset=client.create_dataset(dataset_name)
client.create_examples(
    dataset_name=dataset_name,
    examples = [
        {
            "inputs": {
                "messages": [{"role": "user", "content": "What is the weather like in New York?"}]
            },
            "outputs": {
                "reference": "Agent should call get_weather(city='New York') and summarize sunny result."
            },
            "metadata": {"category": "weather", "difficulty": "easy"},
        },
        {
            "inputs": {
                "messages": [{"role": "user", "content": "What's the weather in London and Paris?"}]
            },
            "outputs": {
                "reference": "Agent may call get_weather twice or ask to pick one city."
            },
            "metadata": {"category": "weather", "difficulty": "medium"},
        },
        {
            "inputs": {
                "messages": [{"role": "user", "content": "Tell me a joke."}]
            },
            "outputs": {
                "reference": "Short joke; no weather tool required."
            },
            "metadata": {"category": "chitchat", "difficulty": "easy"},
        },
        {
            "inputs": {
                "question": "Transfer $50 to account ACC-999"  # flat key also works if your target fn expects it
            },
            "outputs": {
                "reference": "If HITL enabled, should interrupt on transfer_money before execution."
            },
            "metadata": {"category": "hitl", "difficulty": "hard"},
        }
    ]

)

{'example_ids': ['7c017757-95e6-449a-83c3-61a63109e5fe',
  'd0164c3a-8fe8-4aa4-8606-4ada02a2ff96',
  '0002639a-7460-45cb-8085-0d64479e0302',
  '71b09b96-46a1-4c8c-bf66-f16b82e25ebd'],
 'count': 4,
 'as_of': '2026-09-22T11:01:10.981816164Z'}

### Define Metrics (LLM as a Judge)

In [21]:
from openai import OpenAI
from langsmith import wrappers,traceable,evaluate

# LangSmith traces every judge LLM call
groq_client = wrappers.wrap_openai(
    OpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url="https://api.groq.com/openai/v1",
    )
)


JUDGE_MODEL = "openai/gpt-oss-20b"  # or "openai/gpt-oss-20b"


In [15]:
import json

def groq_judge(system_prompt: str, user_prompt: str, schema_hint: str) -> dict:
    """Call wrapped Groq client; return parsed JSON from the judge."""
    response = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": system_prompt + "\nRespond with JSON only. Schema: " + schema_hint,
            },
            {"role": "user", "content": user_prompt},
        ],
    )
    return json.loads(response.choices[0].message.content)

### Run Evaluations

Check rag_faithfulness, helpfulness, correctness
Correctness==> Response vs Reference answer

Goal==> Measure "How similar/correct is the RAG chain anser, relative to a ground-truth answer"

Mode==> Require a ground truth (reference) answer supplied through a dataset

Evaluator==> Now todays LLM is more powerful so why not we Use LLM-AS-JUDGE to answer correctness

For that each time we use groq_judge function

In [16]:
def helpfulness(inputs: dict, outputs: dict) -> dict:
    """Metric: how helpful is the answer? Score 0–1."""
    question = inputs.get("question", "")
    answer = outputs.get("answer", "")

    result = groq_judge(
        system_prompt="You are an expert evaluator. Score helpfulness from 0 to 1.",
        user_prompt=f"Question:\n{question}\n\nAnswer:\n{answer}",
        schema_hint='{"score": number, "comment": string}',
    )
    return {
        "key": "helpfulness",
        "score": float(result["score"]),
        "comment": result.get("comment", ""),
    }


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Metric: semantic match vs reference answer (needs dataset reference outputs)."""
    result = groq_judge(
        system_prompt=(
            "Compare actual answer to expected answer. "
            "Return score 1 if semantically correct else 0."
        ),
        user_prompt=(
            f"Question: {inputs.get('question')}\n"
            f"Expected: {reference_outputs.get('answer')}\n"
            f"Actual: {outputs.get('answer')}"
        ),
        schema_hint='{"score": 0 or 1, "comment": string}',
    )
    return {
        "key": "correctness",
        "score": float(result["score"]),
        "comment": result.get("comment", ""),
    }


def rag_faithfulness(inputs: dict, outputs: dict) -> dict:
    """Metric: is the answer grounded in retrieved context? (good for RAG)."""
    result = groq_judge(
        system_prompt=(
            "Check if the answer is supported by the context only. "
            "Score 0–1 (1 = fully grounded, 0 = hallucinated)."
        ),
        user_prompt=(
            f"Question: {inputs.get('question')}\n"
            f"Context: {outputs.get('context', '')}\n"
            f"Answer: {outputs.get('answer')}"
        ),
        schema_hint='{"score": number, "comment": string}',
    )
    return {
        "key": "faithfulness",
        "score": float(result["score"]),
        "comment": result.get("comment", ""),
    }


def relevance(inputs: dict, outputs: dict) -> dict:
    """Metric: does the answer address the question? Score 0–1."""
    result = groq_judge(
        system_prompt=(
            "Score how relevant the answer is to the question. "
            "1 = fully on-topic, 0 = unrelated."
        ),
        user_prompt=(
            f"Question: {inputs.get('question')}\n"
            f"Answer: {outputs.get('answer')}"
        ),
        schema_hint='{"score": number, "comment": string}',
    )
    return {
        "key": "relevance",
        "score": float(result["score"]),
        "comment": result.get("comment", ""),
    }


def groundedness(inputs: dict, outputs: dict) -> dict:
    """Metric: is the answer supported by the provided context? Score 0–1."""
    result = groq_judge(
        system_prompt=(
            "Score groundedness: is every claim in the answer supported by the context? "
            "1 = fully grounded, 0 = unsupported or hallucinated. "
            "If context is empty, score based on whether the answer avoids invented specifics."
        ),
        user_prompt=(
            f"Question: {inputs.get('question')}\n"
            f"Context: {outputs.get('context', '')}\n"
            f"Answer: {outputs.get('answer')}"
        ),
        schema_hint='{"score": number, "comment": string}',
    )
    return {
        "key": "groundedness",
        "score": float(result["score"]),
        "comment": result.get("comment", ""),
    }

### App under test + dataset + run eval

In [ ]:
### @traceable is decorator that trace your function automatically
@traceable
def chatbot_app(inputs: dict) -> dict:
    """Replace with your real chatbot / Optimize RAG call."""
    question = inputs["question"]
    # Example stub — wire to search_and_answer(...) from Optimize RAG
    return {"answer": f"Stub response for: {question}", "context": ""}


dataset_name = "Chatbot Evaluation Process Test/check"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,  # not dataset_is
    examples=[
        {
            "inputs": {"question": "What is the weather like in New York?"},
            "outputs": {"answer": "Agent should call get_weather(city='New York') and summarize sunny result.."},
        },
        {
            "inputs": {"question": "What's the weather in London and Paris?"},
            "outputs": {"answer": "Agent may call get_weather twice or ask to pick one city."},
        },
    ],
)

results = evaluate(
    chatbot_app,
    data=dataset_name,
    evaluators=[
        helpfulness,
        correctness,       # uses reference_outputs from examples
        rag_faithfulness,  # optional for RAG apps
        relevance,
        groundedness,
    ],
    experiment_prefix="groq-llm-judge",
)

results

d:\work_dsi\LangChain\VTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'groq-llm-judge-05bb3cb1' at:
https://smith.langchain.com/o/014802de-0e48-470f-a47c-836aa450943b/datasets/05dd9014-91b8-44be-a033-4ed2cc6dff8d/compare?selectedSessions=510aa294-97a3-4a27-96de-99b7c5a65ec6




2it [00:04,  2.25s/it]


,inputs.question,outputs.answer,outputs.context,error,reference.answer,feedback.helpfulness,feedback.correctness,feedback.faithfulness,execution_time,example_id,id
0,What is the weather like in New York?,Stub response for: What is the weather like in...,,None,Agent should call get_weather(city='New York')...,0.0,0.0,0.0,0.035999,80313f1a-922f-42d5-9d95-16994013a082,01a0c8f9-9fab-7692-bbbf-966d29b77a6b
1,What's the weather in London and Paris?,Stub response for: What's the weather in Londo...,,None,Agent may call get_weather twice or ask to pic...,0.0,0.0,1.0,0.000000,cf0b9d9c-e7b3-4dc5-9be2-dc25d876a73b,01a0c8f9-a882-7ee0-aecc-8d24ffef5229


### Dataset 2: LLM fundamentals + evaluation (relevance, groundedness)

In [ ]:
import sys
from pathlib import Path

_rag_eval = Path.cwd() if (Path.cwd() / "eval_metrics.py").exists() else Path.cwd() / "RAG_Evaluation"
sys.path.insert(0, str(_rag_eval))
from eval_metrics import helpfulness, correctness, relevance, groundedness

dataset_name_2 = "LLM Fundamentals Evaluation"
dataset_2 = client.create_dataset(dataset_name_2)

client.create_examples(
    dataset_id=dataset_2.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
    ],
)

results_2 = evaluate(
    chatbot_app,
    data=dataset_name_2,
    evaluators=[
        helpfulness,
        correctness,
        relevance,
        groundedness,
    ],
    experiment_prefix="groq-llm-judge-fundamentals",
)

results_2